In [82]:
import sys
sys.path.append('../')
from src.core.scraper.app import ScrapingUtils
from src.core.scraper.processor import ImagesProcessor
images_processor = ImagesProcessor()
scraper_utils = ScrapingUtils()

In [83]:
url = "https://aktmotos.com/motos-akt/automaticas/jet-evo/"

In [84]:
result = images_processor.test_extract(url,["html"])
# print(result.html)
# model_data = images_processor.get_model_data(url=url)

### Manejo con Bs4

In [ ]:
from bs4 import BeautifulSoup

html = result.html

soup = BeautifulSoup(html, "html.parser")
specs_div = soup.find("div", id="contenedor-rotador-1")

# Extraer el valor de total-images del tag image-rotator
image_rotator = specs_div.find("image-rotator") if specs_div else None
total_images = image_rotator.get("total-images") if image_rotator else None # 8
uri_pattern = image_rotator.get("src") if image_rotator else None # /wp-content/uploads/2026/01/akt-jet-evo-negro-01.webp

In [67]:
# Apartir de acá separaremos texto y encontraremos patrones. Lo que buscamos es poder quitar el consecutivo final:
# url original: 'akt-jet-evo-negro-01.webp'
# objetivo:
#   - uri base: /wp-content/uploads/2026/01/
#   - model_name_uri: akt-jet-evo-negro
#   - extension: webp
# url objetivo: https://aktmotos.com/{url_base}/{model_name_uri}/0{i}.{extension}
text_to_extract_extension = uri_pattern.split("/")[-1] # 'akt-jet-evo-negro-01.webp'

In [68]:
uri_base = uri_pattern.replace(text_to_extract_extension, "") # /wp-content/uploads/2026/01/ <-- Apartir de acá armaremos la URL
model_name_uri = text_to_extract_extension.split(".")[0] # akt-jet-evo-negro-01 <-- Se tiene parte del nombre de la imagen
model_name = model_name_uri.replace("-01", "") # akt-jet-evo-negro <-- Se quita el consecutivo final # ! Posiblemente no todos inicien con -01
extension = text_to_extract_extension.split(".")[-1] # webp <-- Usada para completar la extensión de la URL

In [69]:
uri_base

'/wp-content/uploads/2026/01/'

In [70]:
model_name

'akt-jet-evo-negro'

In [71]:
extension

'webp'

In [75]:
total_images

'8'

In [80]:

image_list = []
for i in range(0,int(total_images)):
    image_list.append(f"https://aktmotos.com/{uri_base}{model_name}-0{i+1}.{extension}")

In [81]:
image_list

['https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-01.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-02.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-03.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-04.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-05.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-06.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-07.webp',
 'https://aktmotos.com//wp-content/uploads/2026/01/akt-jet-evo-negro-08.webp']

In [7]:
for image in result.images:
    # Con esta se arma la url base
    if "interna-de-producto" in image:
        url_base_to_process = image # 'https://media.autecomobility.com/recursos/marcas/tvs/raider-125/interna-de-producto/Imagen_Fondo_Texto_detalle_2_TVS.webp'
        break # Solo con la primera coincidencia sirve

# Se separa por barras (/) y se elimina el último elemento para crear así la url base
text_to_eliminate = url_base_to_process.split("/")[-1] # /interna-de-producto/'
url_base = url_base_to_process.replace(text_to_eliminate, "") # 'https://media.autecomobility.com/recursos/marcas/tvs/raider-125/interna-de-producto/'

TypeError: 'NoneType' object is not iterable

In [ ]:
url_base_list = []
for url in range(0,7):
    url_base_list.append(f"{url_base}/Galeria-imagen-{url+1}")

url_base_list

In [ ]:
import requests
default_extension = "webp"
alt_extension = "png"
url_list_checked = []
# Itera entre cada url sin extensión y agrega la extensión por defecto
for url in url_base_list:
    url_to_check = f"{url}.{default_extension}"
    response = requests.get(url_to_check) # Se hace un reuquest para comprobar el status_code que retorne
    if response.status_code == 404: # Si el status_code es 404, se agrega la extensión alternativa
        response = requests.get(f"{url}.{alt_extension}") # Se vuelve a hacer el request con la extensión alternativa
        if response.status_code == 200: # Si el status_code es 200, se agrega la url con la extensión alternativa
            url_list_checked.append(f"{url}.{alt_extension}")
    if response.status_code == 200: # Si el status_code es 200, se agrega la url con la extensión alternativa
        url_list_checked.append(f"{url}.{alt_extension}")

In [ ]:
url_list_checked

In [ ]:
# import requests
# default_extension = "webp"
# alt_extension = "png"
# url_list_checked = []
# # Itera entre cada url sin extensión y agrega la extensión por defecto
# for url in url_base_list:
#     print(f"Probando con extensión por defecto: {url}.{default_extension}")
#     response = requests.get(f"{url}.{default_extension}") # Se hace un reuquest para comprobar el status_code que retorne
#     if response.status_code == 404: # Si el status_code es 404, se agrega la extensión alternativa
#         print(f"Probando con extensión alternativa: {url}.{alt_extension}")
#         response = requests.get(f"{url}.{alt_extension}") # Se vuelve a hacer el request con la extensión alternativa
#         if response.status_code == 200: # Si el status_code es 200, se agrega la url con la extensión alternativa
#             url_list_checked.append(f"{url}.{alt_extension}")
#     url_list_checked.append(f"{url}.{alt_extension}")


In [ ]:
url_list_checked

In [ ]:
result.images

In [ ]:

#
urls_images_list = []
for image in result.images:
    if "width=800" in image:
        urls_images_list.append(image)